# D112 — PIP Package Management

`pip` downloads, installs, upgrades, inspects, and removes Python distributions. Run pip through the intended Python environment:

```bat
C:\Users\GOPALAKRISHNANSUBRAM\dataeng\Scripts\activate.bat
python -m pip --version
```

`python -m pip` is safer than a bare `pip` command because it clearly uses the activated Python.

## 1. Confirm the environment

The pip location should belong to `C:\Users\GOPALAKRISHNANSUBRAM\dataeng`.

In [ ]:
import sys
import subprocess

print("Python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "--version"], check=True)

## 2. Basic install commands

```bat
python -m pip install simplejson
python -m pip install simplejson==3.19.3
python -m pip install "simplejson>=3.19,<4"
python -m pip install --upgrade simplejson
python -m pip uninstall simplejson
```

- No version means pip chooses the newest compatible release.
- `==` pins one exact version.
- `>=` and `<` specify an allowed range. Quote expressions containing `<` or `>` in Command Prompt.
- `--upgrade` replaces an older installed version with the newest compatible release.
- `uninstall` asks for confirmation; add `-y` for non-interactive use.

## 3. Install an older version of `simplejson`

This intentionally installs a fixed release for a reproducible example.

In [ ]:
%pip install "simplejson==3.19.3"

In [ ]:
import simplejson

print("Version:", simplejson.__version__)
print("Location:", simplejson.__file__)
print(simplejson.dumps({"course": "DataEng", "pip": True}, indent=2))

## 4. Find versions and upgrade

Do not assume a version number is still the newest. Ask the configured package index, then upgrade:

```bat
python -m pip index versions simplejson
python -m pip install --upgrade simplejson
python -m pip show simplejson
```

`pip index` is useful for exploration. Production projects should record an approved version instead of upgrading blindly.

In [ ]:
%pip install --upgrade simplejson

In [ ]:
# Reload because simplejson was already imported before the upgrade.
import importlib
simplejson = importlib.reload(simplejson)

print("Version after upgrade:", simplejson.__version__)
print("Location:", simplejson.__file__)

A kernel restart is the safest choice after upgrading a package containing native code. Reloading cannot replace native libraries already loaded into the process.

## 5. Inspect installed packages

```bat
python -m pip list
python -m pip list --outdated
python -m pip show simplejson
python -m pip check
```

- `list` shows installed distributions.
- `list --outdated` compares installed and available versions.
- `show` reports version, location, and declared dependencies.
- `check` reports missing or incompatible installed dependencies.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "show", "simplejson"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

## 6. Generate `requirements.txt`

`freeze` records exact versions of everything installed in the active environment:

```bat
python -m pip freeze > C:\tmp\requirements-full.txt
type C:\tmp\requirements-full.txt
```

A full freeze can include Jupyter and unrelated teaching libraries. For a small application, a curated requirements file is often clearer.

In [ ]:
from pathlib import Path

temp_dir = Path(r"C:\tmp")
temp_dir.mkdir(parents=True, exist_ok=True)

full_requirements = temp_dir / "requirements-full.txt"
with full_requirements.open("w", encoding="utf-8") as output:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=output,
        text=True,
        check=True,
    )

print(full_requirements)
print("Lines:", len(full_requirements.read_text(encoding="utf-8").splitlines()))

In [ ]:
curated_requirements = temp_dir / "requirements.txt"
curated_requirements.write_text(
    "simplejson==3.19.3\n"
    "requests>=2.31,<3\n",
    encoding="utf-8",
)

print(curated_requirements.read_text(encoding="utf-8"))

Common requirement specifiers:

```text
simplejson==3.19.3       # exact pin
requests>=2.31,<3       # compatible range
colorama~=0.4.6         # >=0.4.6 and ==0.4.*
pytest; python_version >= "3.10"  # environment marker
```

Comments begin with `#`. One requirement is normally written per line.

## 7. Reinstall from `requirements.txt`

Install the recorded dependencies into a new or activated environment:

```bat
C:\Users\GOPALAKRISHNANSUBRAM\dataeng\Scripts\activate.bat
python -m pip install -r C:\tmp\requirements.txt
```

Force downloading and reinstalling even when the same versions already exist:

```bat
python -m pip install --force-reinstall -r C:\tmp\requirements.txt
```

For a truly clean reproduction, create a fresh virtual environment instead of force-reinstalling into a busy environment.

## 8. Requirements versus constraints

A requirements file says what to install. A constraints file limits versions but does not request installation by itself.

```text
:: C:\tmp\constraints.txt
simplejson==3.19.3
```

```bat
python -m pip install -r C:\tmp\requirements.txt -c C:\tmp\constraints.txt
```

Constraints are useful when several projects must obey centrally approved versions.

## 9. Pip cache commands

Pip caches downloads so future installations can reuse them.

```bat
python -m pip cache dir
python -m pip cache info
python -m pip cache list
python -m pip cache remove simplejson
python -m pip cache purge
```

`cache remove simplejson` removes cached wheel files matching that distribution. `cache purge` removes **all files from pip's wheel and HTTP caches**. It does not uninstall packages, delete the virtual environment, remove source code, or delete `requirements.txt`.

Purge only reclaims cache space; later installs may need to download the files again.

In [ ]:
# Inspection only. The notebook does not purge automatically.
subprocess.run([sys.executable, "-m", "pip", "cache", "dir"], check=True)
subprocess.run([sys.executable, "-m", "pip", "cache", "info"], check=True)

To bypass the cache for one operation:

```bat
python -m pip install --no-cache-dir simplejson
```

This avoids reading and writing pip's cache for that command; it does not clean an existing cache.

## 10. Useful pip details

Preview dependency resolution without installing:

```bat
python -m pip install --dry-run simplejson
python -m pip install --dry-run --report C:\tmp\pip-report.json -r C:\tmp\requirements.txt
```

Download now and install later or offline:

```bat
mkdir C:\tmp\wheelhouse
python -m pip download -r C:\tmp\requirements.txt -d C:\tmp\wheelhouse
python -m pip install --no-index --find-links C:\tmp\wheelhouse -r C:\tmp\requirements.txt
```

Install a local project in editable development mode:

```bat
python -m pip install -e C:\path\to\project
```

Show detailed resolver output when diagnosing a failure:

```bat
python -m pip install -v simplejson
python -m pip debug --verbose
```

## 11. Safe working habits

1. Activate the intended virtual environment.
2. Confirm with `python -m pip --version`.
3. Use exact pins when identical environments are required.
4. Review `pip install --dry-run` and `pip check` results.
5. Keep requirements files with the project; do not rely on the pip cache as a package backup.
6. Avoid `--user` inside a virtual environment and avoid running pip as Administrator unless genuinely required.
7. Prefer a fresh environment when verifying that a requirements file is complete.

### Summary

- Install: `python -m pip install package`
- Pin: `python -m pip install package==version`
- Upgrade: `python -m pip install --upgrade package`
- Export: `python -m pip freeze > requirements.txt`
- Recreate: `python -m pip install -r requirements.txt`
- Validate: `python -m pip check`
- Inspect cache: `python -m pip cache info`
- Clear all cached downloads: `python -m pip cache purge`